In [18]:
import pandas as pd
from pathlib import Path
from sklearn.metrics import f1_score

In [19]:
pd.set_option('future.no_silent_downcasting', True)

## Pathing

In [20]:
base_path = Path.home() / "kg_aug_causal_disc_exp"

In [21]:
expert_edges_path = base_path / "data" / "expert_edge_pairs.csv"
rag_path = base_path / "results" / "rag.csv"
kgrag_path = base_path / "results" / "kgrag.csv"

In [22]:
expert_edges = pd.read_csv(expert_edges_path)
rag = pd.read_csv(rag_path, index_col=0)
kgrag = pd.read_csv(kgrag_path, index_col=0)
rag = rag.drop(columns=["Label"])
kgrag = kgrag.drop(columns=["Label"])

In [6]:
print(kgrag[(kgrag["Var1"] == "Age") & (kgrag["Var2"] == "Alcohol")]["Association Report"].iloc[0])

# Report:
## Chunks:

0"></span>heterogeneity test, Supplementary Table 3); otherwise, a fixedeffect IVW was used. The weighted median method was also used since it can provide consistent estimates when up to 50% of the weight in the analysis were originated from invalid instrumental variables (28). In addition, in case the instrumental variable assumptions cannot be fully satisfied, we also performed the MR-Egger method and MR-PRESSO (Mendelian Randomization Pleiotropy RESidual Sum and Outlier) method to detect potential pleiotropy and outliers (29, 30). The MR-Egger method can identify and correct potential pleiotropy (p for intercept < 0.05) and gives a consistent estimate (29). The outlier test in the MR-PRESSO can detect possible outliers and provide adjusted results after excluding the outliers and thus correcting for the horizontal pleiotropy. Next, to assess whether the causal effects of other obesity-related traits on the outcomes were mediated by BMI, we performed multivariab

In [7]:
print(rag[(rag["Var1"] == "Age") & (rag["Var2"] == "Alcohol")]["Association Report"].iloc[0])

 binge drinking within the past month and 85.6% of adults report trying alcohol during their lifetime. Of those individuals aged 12 and older, 14.5 million met criteria for Alcohol Use Disorder . A substance use disorder, sometimes referred to as substance abuse, refers to a clinical level of impairment experienced by an individual in domains such as work, school, and home, due to use of alcohol and/or drugs (American Psychiatric Association; APA, 2013). Of note, use, such as experimentation, with alcohol or another substance does not necessarily indicate a level of dependency associated with an alcohol or substance use disorder/abuse. Further, 'abuse' often coincides with serious medical impairment and risk (e.g. cirrhosis of the liver) whereas 'use' may increase or decrease (i.e. lack of dependency) and may occur with less severe health risks (Cicchetti & Handley, 2019; Hendler & Stephens, 1977; Roerecke et al., 2019). Addiction refers to an individual's state of dependence either ps

In [8]:
print(rag[(rag["Var1"] == "Age") & (rag["Var2"] == "Alcohol")]["Association Reasoning"].iloc[0])

Reasoning Process:
Step 1: The report mentions that 'Of those individuals aged 12 and older, 14.5 million met criteria for Alcohol Use Disorder.' This implies that there is a statistical association between age and alcohol, as the number of individuals with alcohol use disorder varies with age.
Step 2: The report also mentions that 'Most states also prohibit underage consumption (i.e., consumption of alcoholic beverages prior to the age of 21).' This suggests that there is a restriction or rule in place that is based on age, indicating a relationship between age and alcohol consumption.

Conclusion: Yes


In [9]:
expert_edges["Label"] = True

In [10]:
expert_edges

,Var1,Var2,Label
0,Age,CCI,True
1,Age,Obesity,True
2,Age,Anxiety,True
3,Age,Depression,True
4,Age,Sleep disturbance,True
...,...,...,...
59,Sleep disturbance,CCI,True
60,Sleep disturbance,Depression,True
61,Sleep disturbance,Smoking,True
62,Sleep disturbance,PEG,True


In [11]:
rag = rag.merge(expert_edges, how="outer", on=["Var1", "Var2"])
kgrag = kgrag.merge(expert_edges, how="outer", on=["Var1", "Var2"])
rag["Label"] = rag["Label"].fillna(False)
kgrag["Label"] = kgrag["Label"].fillna(False)
rag["Label"] = rag["Label"].astype(bool)
kgrag["Label"] = kgrag["Label"].astype(bool)

In [12]:
f1_score(rag["Label"], rag["Plausibility"])

0.6116504854368932

In [13]:
f1_score(kgrag["Label"], kgrag["Plausibility"])

0.6019417475728155

In [14]:
f1_score(rag["Label"], rag["Association"])

0.5578231292517006

In [15]:
f1_score(kgrag["Label"], kgrag["Association"])

0.4032258064516129

In [16]:
f1_score(rag["Label"], rag["Temporality"])

0.42201834862385323

In [17]:
f1_score(kgrag["Temporality"], kgrag["Label"])

0.2558139534883721

In [18]:
kgrag["Plausibility"].value_counts()

Plausibility
True     142
False      3
Name: count, dtype: int64

In [19]:
rag["Plausibility"].value_counts()

Plausibility
True     142
False      3
Name: count, dtype: int64

In [20]:
kgrag["Association"].value_counts()

Association
False    85
True     60
Name: count, dtype: int64

In [21]:
rag["Association"].value_counts()

Association
True     83
False    62
Name: count, dtype: int64

In [22]:
rag["Temporality"].value_counts()

Temporality
False    100
True      45
Name: count, dtype: int64

In [23]:
kgrag["Temporality"].value_counts()

Temporality
False    123
True      22
Name: count, dtype: int64

In [24]:
metric = "Plausibility"

rag_edges = rag[rag[metric]][["Var1", "Var2"]]
kgrag_edges = kgrag[kgrag[metric]][["Var1", "Var2"]]
diff = rag_edges.merge(kgrag_edges, how="outer",indicator=True)
diff[diff['_merge'] == 'left_only']
diff["_merge"].value_counts()

_merge
both          140
left_only       2
right_only      2
Name: count, dtype: int64